# LangGraph & MCP – Stateful Factory Agents

**Internet of Production** · Prof. Dr.-Ing. U. Triltsch · Ostfalia University of Applied Sciences

---

## Learning objectives

After completing this notebook you are able to:

1. Explain the core concepts of **LangGraph** (state graph, nodes, edges, checkpointing).
2. Map the familiar MESA factory scenario (`MachineAgent ↔ CoordinatorAgent ↔ MaintenanceAgent`) onto a LangGraph workflow.
3. Understand the role of **MCP (Model Context Protocol)** as a connectivity layer between LLM-based agents and external tools or data sources.
4. Distinguish rule-based reactive behaviour (MESA) from LLM-augmented stateful behaviour (LangGraph).

---

## Classification: where does this notebook fit?

| Notebook | Agent paradigm | Communication | LLM? |
|---|---|---|---|
| `01_Reactive_Agents_solution.ipynb` | Reactive (sense–decide–act) | Local grid state | No |
| `04_Reactive_Agents_OPC_solution.ipynb` | Communicative | OPC UA (shared bus) | No |
| **This notebook** | **Stateful / LLM-augmented** | **Shared graph state + MCP** | **Yes (optional)** |

We use the same **factory scenario**: machines heat up, the coordinator monitors HOT states, the maintenance agent repairs machines. What changes is the *architectural pattern* — from the MESA simulation loop to a **LangGraph state machine**.


## 1. Conceptual foundations

### 1.1 LangGraph — core ideas

LangGraph models a multi-agent workflow as a **directed graph**:

```
Node A ──(edge condition)──► Node B ──► Node C
        ▲                                 │
        └────────────(loop back)──────────┘
```

- **State**: a typed dictionary shared by all nodes (analogous to the OPC UA address space from the previous notebooks).
- **Nodes**: Python functions (or LLM-based agents) that *read* the state, do work and *write back* updates.
- **Edges**: routing logic — which node runs next? It can be conditional (corresponds to reactive rules).
- **Checkpointing**: the state is persistable; the graph can be interrupted and resumed (human-in-the-loop).

### 1.2 MCP — Model Context Protocol

MCP is the standardised **connectivity layer** between agents and external tools:

```
 LLM agent (host)
      │
   MCP client ──JSON-RPC 2.0──► MCP server: OPC UA   ──► PLC / SCADA
   MCP client ──────────────► MCP server: REST API   ──► ERP / MES
   MCP client ──────────────► MCP server: database   ──► TimescaleDB
```

Key message: the agent needs **no integration code of its own** per tool — it calls a standardised MCP interface. The server side encapsulates the protocol-specific logic (OPC UA, REST, SQL).

### 1.3 Conceptual mapping: MESA → LangGraph

| MESA concept | LangGraph equivalent |
|---|---|
| `agent.step()` | Graph node function |
| `model.grid` / `model.schedule` | Shared `GraphState` dict |
| `DataCollector` | State fields (e.g. `hot_machines`) |
| OPC UA subscription callback | Conditional edge routing |
| `MaintenanceAgent.choose_target()` | CoordinatorAgent node logic |
| Mesa `RandomActivation` scheduler | LangGraph execution order |


## 2. Installation

LangGraph is part of the LangChain ecosystem. We install only the minimal dependencies for this example (no external LLM API key is required for the rule-based variant).


In [ ]:
# Install LangGraph and LangChain Core
!pip install -q "langgraph>=0.2" "langchain-core>=0.2"

In [ ]:
import random
import time
from typing import TypedDict, List, Dict, Literal

from langgraph.graph import StateGraph, END

print("LangGraph import successful.")

## 3. Defining the shared graph state

The `GraphState` is the **single source of truth** for all agents in the graph —
analogous to the OPC UA address space in the notebook on communicative agents.

Every node receives the complete state and returns a *partial update* (only the fields it changes).


In [ ]:
class MachineStatus(TypedDict):
    """Machine status entry in the shared state."""
    name: str
    temperature: float
    state: str          # "OK" | "WARM" | "HOT"
    repair_needed: bool


class GraphState(TypedDict):
    """Shared state for the factory agent graph.
    
    All agents read from and write to this dictionary.
    LangGraph merges partial updates automatically.
    """
    # Machine telemetry (updated by the MachineAgent node)
    machines: Dict[str, MachineStatus]
    
    # Coordinator decisions (updated by the CoordinatorAgent node)
    repair_queue: List[str]        # list of machine names waiting for repair
    coordinator_log: List[str]     # audit log of coordinator decisions
    
    # Maintenance progress (updated by the MaintenanceAgent node)
    maintenance_log: List[str]
    
    # Control fields
    cycle: int                     # simulation step counter
    max_cycles: int
    next_action: str               # evaluated by the router edge


# ── Helper function: initial state for N machines ────────────────────────────
def initial_state(n_machines: int = 5, max_cycles: int = 10) -> GraphState:
    machines = {}
    for i in range(n_machines):
        name = f"M{i+1:02d}"
        machines[name] = MachineStatus(
            name=name,
            temperature=20.0,
            state="OK",
            repair_needed=False,
        )
    return GraphState(
        machines=machines,
        repair_queue=[],
        coordinator_log=[],
        maintenance_log=[],
        cycle=0,
        max_cycles=max_cycles,
        next_action="machine_node",
    )

print("GraphState defined.")
print("Example initial state for 3 machines:")
s = initial_state(3, max_cycles=5)
for k, v in s.items():
    print(f"  {k}: {v}")

## 4. Agent nodes

Each agent is implemented as a **simple Python function** that:
1. receives the current `GraphState`,
2. executes its logic and
3. returns a *partial state update* (only the changed fields).

This pattern is identical for all LangGraph nodes — regardless of whether they are rule-based (as here) or LLM-based.

### 4.1 MachineAgent node

Simulates the temperature measurement for all machines (corresponds to `MachineAgent.sense_temperature()` + `MachineAgent.decide()` in the MESA notebooks).


In [ ]:
THRESHOLD = 70.0  # °C


def machine_node(state: GraphState) -> dict:
    """MachineAgent node: measure temperature and update the state of all machines.
    
    Corresponds to the MESA method MachineAgent.step(), but applied to all
    machines in a single node execution.
    """
    updated_machines = {}
    
    for name, m in state["machines"].items():
        temp = m["temperature"]
        
        # Sense: add random thermal drift (simulated sensor measurement)
        if m["state"] != "HOT":
            noise = random.uniform(-1.0, 5.0)
            temp = temp + noise
        
        # Decide: classify the state based on the threshold
        if temp > THRESHOLD:
            new_state = "HOT"
        elif temp > 0.75 * THRESHOLD:
            new_state = "WARM"
        elif temp < 20.0:
            new_state = "COOL"
        else:
            new_state = "OK"
        
        updated_machines[name] = MachineStatus(
            name=name,
            temperature=round(temp, 1),
            state=new_state,
            repair_needed=m["repair_needed"],  # unchanged here
        )
        print(f"  [MachineAgent] {name}: {round(temp,1)} °C → {new_state}")
    
    return {"machines": updated_machines, "next_action": "coordinator_node"}


print("machine_node defined.")

### 4.2 CoordinatorAgent node

Reads all machine states from the shared state, detects HOT machines and fills the `repair_queue`. This is the *orchestration logic* — in a production system this node could call an LLM to justify the prioritisation (see section 6).

Note the analogy to the function `sync_repair_jobs_from_opcua()` in the notebook on communicative agents: there, too, the coordinator acts as a bridge between sensing and action.


In [ ]:
def coordinator_node(state: GraphState) -> dict:
    """CoordinatorAgent node: detect HOT machines and manage the repair queue.
    
    Decision rules:
    - Add HOT machines to the repair_queue if not already present.
    - Remove machines from the queue when they are no longer HOT.
    - Write a log entry for every decision.
    """
    cycle = state["cycle"]
    existing_queue = list(state["repair_queue"])
    log = list(state["coordinator_log"])
    machines = state["machines"]
    
    new_queue = []
    
    # Remove from the queue if no longer HOT (already repaired)
    for name in existing_queue:
        if machines[name]["state"] == "HOT":
            new_queue.append(name)
        else:
            log.append(f"[Cycle {cycle}] [COORD] {name} removed from the repair queue.")
    
    # Add machines that have newly become HOT
    for name, m in machines.items():
        if m["state"] == "HOT" and name not in new_queue:
            new_queue.append(name)
            log.append(
                f"[Cycle {cycle}] [COORD] {name} at {m['temperature']} °C → REPAIR SCHEDULED."
            )
            print(f"  [CoordinatorAgent] {name} HOT → added to the repair queue.")
    
    # Update the repair_needed flags in the machine states
    updated_machines = {}
    for name, m in machines.items():
        updated_machines[name] = MachineStatus(
            name=m["name"],
            temperature=m["temperature"],
            state=m["state"],
            repair_needed=(name in new_queue),
        )
    
    next_action = "maintenance_node" if new_queue else "advance_cycle"
    
    return {
        "machines": updated_machines,
        "repair_queue": new_queue,
        "coordinator_log": log,
        "next_action": next_action,
    }


print("coordinator_node defined.")

### 4.3 MaintenanceAgent node

Processes the repair queue: resets temperature and state of the first machine in the queue.
In the MESA notebooks the `MaintenanceAgent` moved physically through the grid;
here "movement" and "repair" are combined into a single state update
(the graph abstracts the spatial dimension).


In [ ]:
def maintenance_node(state: GraphState) -> dict:
    """MaintenanceAgent node: repair the first machine in the repair queue.
    
    Simplification for teaching purposes: one repair per graph cycle.
    In a production system several maintenance agents could run in parallel
    (LangGraph multi-actor pattern).
    """
    queue = list(state["repair_queue"])
    machines = dict(state["machines"])
    log = list(state["maintenance_log"])
    cycle = state["cycle"]
    
    if not queue:
        return {"next_action": "advance_cycle"}
    
    # Repair the first machine in the queue
    target = queue.pop(0)
    
    # Reset the machine (corresponds to: agent.temperature = 20.0; agent.state = "OK")
    machines[target] = MachineStatus(
        name=target,
        temperature=20.0,
        state="OK",
        repair_needed=False,
    )
    
    entry = f"[Cycle {cycle}] [MAINT] {target} repaired → temperature reset to 20.0 °C."
    log.append(entry)
    print(f"  [MaintenanceAgent] {entry}")
    
    return {
        "machines": machines,
        "repair_queue": queue,
        "maintenance_log": log,
        "next_action": "advance_cycle",
    }


def advance_cycle_node(state: GraphState) -> dict:
    """Increment the cycle number. Acts as the loop-control node."""
    new_cycle = state["cycle"] + 1
    print(f"\n─── End of cycle {state['cycle']} ───\n")
    return {"cycle": new_cycle, "next_action": "machine_node"}


print("maintenance_node and advance_cycle_node defined.")

## 5. Building the LangGraph state graph

We now assemble the nodes and connect them with **conditional edges**.
The routing function reads `state["next_action"]` — set by each node —
to determine which node runs next.

```
START
  │
  ▼
machine_node  ──────────────────────────────────────────────►
  │                                                    coordinator_node
  │                                                          │
  │                          HOT machines?  ─── YES ──► maintenance_node
  │                                                          │
  │               ◄───────── NO ─── advance_cycle_node ◄─────┘
  │                                              │
  └────────── cycle < max_cycles ◄───────────────┘
                     │
              cycle >= max_cycles
                     │
                    END
```


In [ ]:
# ── Router: reads next_action from the state ─────────────────────────────────
def router(state: GraphState) -> Literal[
    "machine_node", "coordinator_node", "maintenance_node", "advance_cycle", "__end__"
]:
    """Route to the next node based on state['next_action'] and the cycle count."""
    if state["cycle"] >= state["max_cycles"]:
        return "__end__"
    return state["next_action"]


# ── Build the graph ──────────────────────────────────────────────────────────
builder = StateGraph(GraphState)

# Add nodes
builder.add_node("machine_node",     machine_node)
builder.add_node("coordinator_node", coordinator_node)
builder.add_node("maintenance_node", maintenance_node)
builder.add_node("advance_cycle",    advance_cycle_node)

# Entry point
builder.set_entry_point("machine_node")

# Conditional edges (all nodes use the same router)
for node in ["machine_node", "coordinator_node", "maintenance_node", "advance_cycle"]:
    builder.add_conditional_edges(
        node,
        router,
        {
            "machine_node":     "machine_node",
            "coordinator_node": "coordinator_node",
            "maintenance_node": "maintenance_node",
            "advance_cycle":    "advance_cycle",
            "__end__":          END,
        },
    )

# Compile the graph
graph = builder.compile()

print("Graph compiled successfully.")
print("Nodes:", list(graph.nodes))

## 5.1 Running the simulation


In [ ]:
# Initial state: 5 machines, 30 simulation cycles
state = initial_state(n_machines=5, max_cycles=30)

print("=" * 60)
print("FACTORY AGENT SIMULATION — LangGraph")
print(f"Machines: {len(state['machines'])}  |  Max. cycles: {state['max_cycles']}")
print("=" * 60)

# Run the graph until END
final_state = graph.invoke(state)

print("\n" + "=" * 60)
print("SIMULATION FINISHED")
print("=" * 60)

In [ ]:
# ── Show final machine states ─────────────────────────────────────────────────
print("\nFinal machine states:")
print(f"{'Machine':<10} {'Temperature':>13} {'State':<8} {'Repair needed'}")
print("-" * 50)
for name, m in final_state["machines"].items():
    print(f"{name:<10} {m['temperature']:>10.1f} °C {m['state']:<8} {m['repair_needed']}")

# ── Coordinator log ───────────────────────────────────────────────────────────
print("\nCoordinator log:")
for entry in final_state["coordinator_log"]:
    print(" ", entry)

# ── Maintenance log ───────────────────────────────────────────────────────────
print("\nMaintenance log:")
for entry in final_state["maintenance_log"]:
    print(" ", entry)

## 6. Extension: LLM-based CoordinatorAgent (conceptual)

The coordinator node above uses **rule-based logic**. In a production system, replacing it with an LLM call only requires changing the node body:

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

def coordinator_node_llm(state: GraphState) -> dict:
    # Serialise the relevant state for the LLM
    machine_summary = [
        f"{name}: {m['temperature']} °C, state={m['state']}"
        for name, m in state['machines'].items()
    ]
    prompt = (
        "You are a factory coordinator agent. "
        "Based on the following machine telemetry, decide which machines "
        "need to be repaired and justify your prioritisation.\n\n"
        + "\n".join(machine_summary)
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    # Evaluate the LLM response and update the state accordingly ...
    ...
```

**Key observation**: the rest of the graph (`machine_node`, `maintenance_node`, edges) remains unchanged. This is the architectural advantage of LangGraph — nodes are interchangeable independently of each other.

### 6.1 MCP integration (conceptual)

If the coordinator is to read from a real OPC UA server instead of the simulated state, an MCP server that wraps the OPC UA endpoint provides the standardised tool interface:

```python
# Conceptual: MCP tool call inside a LangGraph node
# The agent calls the MCP tool;
# the server handles the OPC UA protocol details.

tools = [
    {
        "type": "mcp_tool",
        "server": "opcua-factory-server",
        "name": "read_machine_temperature",
        "input": {"machine_id": "M01"}
    }
]
# The MCP server translates this into:
#   asyncua.Client.get_node(node_id).read_value()
# ... and returns the result to the agent.
```

This mirrors the architecture shown in the lecture:
**LLM agent (host) → MCP client → MCP server: OPC UA → PLC/SCADA**


## 7. Comparative analysis: MESA vs. LangGraph

| Dimension | MESA (reactive/communicative) | LangGraph |
|---|---|---|
| **State management** | Per-agent attributes + model-wide DataCollector | Central typed `GraphState` dict, persistable |
| **Execution control** | Scheduler (RandomActivation etc.) | Explicit graph topology + conditional edges |
| **Communication** | Shared grid / OPC UA address space | Shared state dict (+ optional MCP for external data) |
| **Observability** | DataCollector → pandas DataFrame | State snapshot at every node; full checkpointing |
| **LLM integration** | Not natively supported | First-class (every node can call an LLM) |
| **Human-in-the-loop** | Not supported | Built in (interrupt before/after every node) |
| **Scalability** | Thousands of agents on one machine | Limited by LLM latency; suitable for ~10–100 agents |
| **Learning curve** | Low (plain Python, MESA API) | Medium (graph abstractions, LangChain ecosystem) |
| **Best use cases** | Population level, spatial dynamics | Orchestration, complex decision logic, LLM reasoning |

**Conclusion**: MESA and LangGraph are complementary, not competing. MESA is the right tool for simulating large populations of simple agents. LangGraph is suitable when individual agents need persistent state, complex decision reasoning, LLM augmentation or human oversight — exactly the requirements of a production-grade IIoT orchestration layer.


## 8. Exercises

### 8.1 Adapt the routing logic
- Extend the `coordinator_node` so that WARM machines are added to a separate *early-warning list* (without triggering an immediate repair).
- Add a new node `alert_node` that issues a warning for WARM machines.
- Update the router accordingly.

### 8.2 Parallel maintenance agents
- Currently one machine is repaired per cycle. Modify `maintenance_node` so that up to two machines are repaired per cycle.
- How does this affect the number of HOT machines over time?

### 8.3 Checkpointing (advanced)
- Install `langgraph[sqlite]` and add a `MemorySaver` checkpointer to the compiled graph.
- Interrupt the simulation after 3 cycles, inspect the checkpoint, change a machine temperature manually and resume the simulation.
- This demonstrates the **human-in-the-loop** pattern from the lecture slides.

```python
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
```

### 8.4 MCP server stub (advanced)
- Implement a minimal Python `FastAPI` server that provides a `GET /temperature/{machine_id}` endpoint.
- Modify the `machine_node` so that it calls this HTTP endpoint instead of the local noise simulation.
- Discuss: how would you replace this REST call by an OPC UA read access via an MCP server?


## 9. Summary

This notebook has shown:

1. The **LangGraph state graph** as a production-ready alternative to the MESA simulation loop for agent orchestration.
2. **Typed shared state** (`GraphState`) as the architectural equivalent of the OPC UA address space — a single source of truth for all agents.
3. **Conditional edge routing** as the graph-level equivalent of reactive rules in MESA agents.
4. **MCP** as a standardised integration layer connecting LLM agents with OPC UA servers, REST APIs and databases — without tool-specific integration code.
5. The **interchangeability principle**: any rule-based node can be replaced by an LLM-based node without changing the surrounding graph structure.

---

**Literature and resources**

- LangGraph documentation: https://langchain-ai.github.io/langgraph/
- Model Context Protocol: https://modelcontextprotocol.io
- Google A2A framework: https://codelabs.developers.google.com/intro-a2a-purchasing-concierge
- n8n workflow automation: https://n8n.io
- MESA multi-agent framework: https://mesa.readthedocs.io/en/stable/
